In [5]:
from typing import TypedDict,Annotated,List,Dict,Any
from dotenv import load_dotenv 
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage , HumanMessage
from langgraph.graph import StateGraph , END, START
import operator
import re

In [3]:
load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
class StartupPitchState(TypedDict):
    raw_idea:str
    target_market:str
    funding_stage:str

    refined_pitch:str

    # parallel
    competitor_analysis:str
    monetization_plan:str
    tech_blueprint:str
    regulatory_report:str


    domain_scores:Annotated[List[Dict[str,Any]],operator.add]
    red_flags:Annotated[List[str],operator.add]
    logs:Annotated[List[str],operator.add]

    final_average_score:float
    investment_verdict:str
    vc_memo:str



In [ ]:
def idea_refiner(state:StartupPitchState)->dict:
    idea=state['raw_idea']
    target_market=state['target_market']
    funding_stage=state['funding_stage']

    messag=[
        SystemMessage(
            content=(
                "You are a Startup Incubator Partner. Convert the raw startup idea into a clear, "
                "structured Pitch Brief covering: "
                "1. Core Problem Statement "
                "2. Target Customer Persona "
                "3. Core Value Proposition "
                "4. Key Assumptions to Validate. Keep it crisp."
            )
        ),
        HumanMessage(
            content=(
                f"IDEA:{idea}\n TAEGET MARKET: {target_market}\n FUNDING SATGE:{funding_stage}"
            )
        )
    ]
    resposne=llm.invoke(messag).content
    return{
        "refined_pitch":resposne,
        "logs":["[idea refined] is completed"]
    }

In [ ]:
def competitor_analyst (state:StartupPitchState):
    refined_pitch=state["refined_pitch"]
    message=[
        SystemMessage(
            content=(
                "You are a Market Intelligence Analyst. Based on the refined pitch, provide:\n"
                "1. Top 3 Direct and Indirect Competitors in the region.\n"
                "2. Market Saturation Threat.\n"
                "3. Defensibility (Moat) Strategy.\n\n"
                "At the very end, you MUST include these two exact lines:\n"
                "RED_FLAG: <one sentence stating the biggest competitive threat>\n"
                "SCORE: <integer from 1 to 10 rating market opportunity>"
            )
        ),
        HumanMessage(
            content=f"refined Pitch:{refined_pitch}"
        )

    ]
    response=llm.invoke(message).content
    return{
        "competitor_analysis":response
    }

In [ ]:
builder=StateGraph(StartupPitchState)

builder.add_node("idea_refiner",idea_refiner)
builder.add_node("cometitor_analyst",competitor_analyst)
builder.add_node("monetization_planner",monetization_planner)
builder.add_node("tech_arthitect",tech_arthitect)
builder.add_node("regulatory_report",regulatory_repoter)
builder.add_node("vc_juge_node",vc_juge_node)